In [2]:
from pathlib import Path
import zipfile

import pandas as pd
import searoute as sr
from geopy.distance import great_circle
import coordinates as coord
import os
from core_modules import core_modules as core

In this code, I assemble the necessary data, clean it, conduct an exploratory data analysis, and then ...

# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [3]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is co2(tons)(tons)(tons)(tons)(tons)(tons)(tons) released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Instead of country names, country codes were used as per [country.txt](https://www.census.gov/foreign-trade/schedules/c/country.txt) file on Census.gov.

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

## Distance

This code calculates distances between ports for countries beyond the ocean(China, India, South Korea, Vietnam), as well as land distance over the US border for Mexico.

Because the ISTHS6M and PORTHS6MM is encoded through fixed-width text, it is difficult to read and convert into dataframes. To solve this, I read it once, then saved the snapshot into Parquet. That way

### ISTHS6M (December 2024 — Pre-Tariff)

In [4]:
_snapshot = Path('data/snapshots/ISTHSM2412.parquet')
if _snapshot.exists():
    df_land_imports_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/ISTHSM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_land_imports_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_land,
                names=coord.names_land,
                dtype={c: str for c in coord.str_cols_land},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_land_imports_2412.to_parquet(_snapshot, index=False)
df_land_imports_2412.head()


,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,AK,2024,12,0,0,0,0,0,...,0,0,20337,20337,0,0,0,0,0,0
1,010121,1220,AR,2024,12,0,0,0,0,0,...,0,0,7500,7500,0,0,0,0,0,0
2,010121,1220,CA,2024,12,0,0,0,0,0,...,0,0,3632,3632,0,0,0,0,0,0
3,010121,1220,DE,2024,12,0,0,0,0,0,...,0,0,2183,2183,0,0,0,0,0,0
4,010121,1220,FL,2024,12,138066,138066,0,0,0,...,0,0,453343,453343,0,0,0,0,0,0


In [5]:
df_land_imports_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1194681 entries, 0 to 1194680
Data columns (total 21 columns):
 #   Column      Non-Null Count    Dtype
---  ------      --------------    -----
 0   commodity   1194681 non-null  str  
 1   cty_code    1194681 non-null  str  
 2   state       1194681 non-null  str  
 3   year        1194681 non-null  str  
 4   month       1194681 non-null  str  
 5   gen_val_mo  1194681 non-null  int64
 6   con_val_mo  1194681 non-null  int64
 7   air_val_mo  1194681 non-null  int64
 8   air_swt_mo  1194681 non-null  int64
 9   ves_val_mo  1194681 non-null  int64
 10  ves_swt_mo  1194681 non-null  int64
 11  cnt_val_mo  1194681 non-null  int64
 12  cnt_swt_mo  1194681 non-null  int64
 13  gen_val_yr  1194681 non-null  int64
 14  con_val_yr  1194681 non-null  int64
 15  air_val_yr  1194681 non-null  int64
 16  air_swt_yr  1194681 non-null  int64
 17  ves_val_yr  1194681 non-null  int64
 18  ves_swt_yr  1194681 non-null  int64
 19  cnt_val_yr  1194681 non-null  in

### ISTHS6M (December 2025 — Post-Tariff)

In [6]:
_snapshot = Path('data/snapshots/ISTHSM2512.parquet')
if _snapshot.exists():
    df_land_imports_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/ISTHSM2512.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_land_imports_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_land,
                names=coord.names_land,
                dtype={c: str for c in coord.str_cols_land},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_land_imports_2512.to_parquet(_snapshot, index=False)
df_land_imports_2512.head()


,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,CA,2025,12,25000,25000,0,0,0,...,0,0,25000,25000,0,0,0,0,0,0
1,010121,1220,CO,2025,12,6000,6000,0,0,0,...,0,0,12000,12000,0,0,0,0,0,0
2,010121,1220,CT,2025,12,0,0,0,0,0,...,0,0,3237,3237,0,0,0,0,0,0
3,010121,1220,FL,2025,12,164429,164429,0,0,0,...,0,0,251214,251214,0,0,0,0,0,0
4,010121,1220,ID,2025,12,0,0,0,0,0,...,0,0,22000,22000,0,0,0,0,0,0


This is the code that could be used to auto-import data. However, the problem with it is that there is poor connection between the API and the website.œ

In [7]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Weight of imports

Similar to UN Comtrade, but focuses on the USA and is more laconic.

[USA Trade Census](https://usatrade.census.gov/data/Perspective60/Browse/browsetables.aspx?utosid=5687ae9fc5be588295a68da15cd2f1cd&cache=tffv5e)
+ Data Source Selection: State Import Data(Harmonized System)
+ Filters:
    + Measures: Vessel SWT and Air SWT(kg) - The gross weight in kilograms of shipments made by seafaring vessel/airplane at customs
        + No data on land transportation there
    + State: All States
    + Commodity: 845011, 845012, 845019 and 845020(washing machines)
    + Country: India, South Korea, Mexico
    + Time: Jan 2025 - Mar 2026(monthly)

In [8]:
washing_machine_trade_filepath = "data/original/State Imports by HS Commodities_v4.csv"
washing_machine_df = pd.read_csv(washing_machine_trade_filepath, index_col=False, header=2)

In [9]:
washing_machine_df.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg),Unnamed: 5
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174,"1,777,528",NaN
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,NaN,"2,459,827",NaN
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,NaN,"2,268,490",NaN
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,NaN,"2,512,291",NaN
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,NaN,"2,541,012",NaN


# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

In [10]:
# this defines the columns in the PORTHS6MM dataframe, which are to be converted to numbers
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]


## Ocean-based Distance/Weight Data (Asian Routes, Pre-Tariff — Dec 2024)

In [11]:
_snapshot = Path('data/snapshots/PORTHS6MM2412.parquet')
if _snapshot.exists():
    df_ocean_routes_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/PORTHS6MM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2412.to_parquet(_snapshot, index=False)


In [12]:
df_ocean_routes_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212596 entries, 0 to 1212595
Data columns (total 20 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   commodity    1212596 non-null  str  
 1   cty_code     1212596 non-null  str  
 2   dist_unlade  1212596 non-null  str  
 3   port_unlade  1212596 non-null  str  
 4   year         1212596 non-null  str  
 5   month        1212596 non-null  str  
 6   gen_val_mo   1212596 non-null  int64
 7   air_val_mo   1212596 non-null  int64
 8   air_swt_mo   1212596 non-null  int64
 9   ves_val_mo   1212596 non-null  int64
 10  ves_swt_mo   1212596 non-null  int64
 11  cnt_val_mo   1212596 non-null  int64
 12  cnt_swt_mo   1212596 non-null  int64
 13  gen_val_yr   1212596 non-null  int64
 14  air_val_yr   1212596 non-null  int64
 15  air_swt_yr   1212596 non-null  int64
 16  ves_val_yr   1212596 non-null  int64
 17  ves_swt_yr   1212596 non-null  int64
 18  cnt_val_yr   1212596 non-null  int64
 19  cnt_swt_yr 

In [13]:
df_ocean_routes_2412["port_full"] = df_ocean_routes_2412["dist_unlade"] + df_ocean_routes_2412["port_unlade"]

df_asian_2412 = df_ocean_routes_2412[df_ocean_routes_2412["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2412[col] = pd.to_numeric(df_asian_2412[col])

df_asian_wash_2412 = df_asian_2412[df_asian_2412["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["port_full"].isin(coord.us_ports)]
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["ves_swt_mo"] > 0]
df_asian_wash_2412["distance"] = df_asian_wash_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2412["co2"] = df_asian_wash_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2412.to_csv("data/intermediate/PORTHS6MM_asian_wash_2412.csv")
df_asian_wash_2412.info()

<class 'pandas.DataFrame'>
Index: 79 entries, 799729 to 800031
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    79 non-null     object 
 1   cty_code     79 non-null     object 
 2   dist_unlade  79 non-null     object 
 3   port_unlade  79 non-null     object 
 4   year         79 non-null     object 
 5   month        79 non-null     object 
 6   gen_val_mo   79 non-null     int64  
 7   air_val_mo   79 non-null     int64  
 8   air_swt_mo   79 non-null     int64  
 9   ves_val_mo   79 non-null     int64  
 10  ves_swt_mo   79 non-null     int64  
 11  cnt_val_mo   79 non-null     int64  
 12  cnt_swt_mo   79 non-null     int64  
 13  gen_val_yr   79 non-null     object 
 14  air_val_yr   79 non-null     object 
 15  air_swt_yr   79 non-null     object 
 16  ves_val_yr   79 non-null     object 
 17  ves_swt_yr   79 non-null     object 
 18  cnt_val_yr   79 non-null     object 
 19  cnt_swt_yr   79 n

In [14]:
df_asian_wash_2412["port_full"].value_counts().sort_index(ascending=False)

port_full
5301    5
5201    3
4909    4
3002    5
3001    6
2904    3
2811    5
2709    6
2704    6
2002    1
1901    2
1803    4
1801    4
1703    6
1601    3
1401    4
1303    5
1003    6
0401    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Post-Tariff — Dec 2025)

In [15]:
_snapshot = Path("data/snapshots/PORTHS6MM2512.parquet")
if _snapshot.exists():
    df_ocean_routes_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2512.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2512.to_parquet(_snapshot, index=False)

df_ocean_routes_2512["port_full"] = df_ocean_routes_2512["dist_unlade"] + df_ocean_routes_2512["port_unlade"]

df_asian_2512 = df_ocean_routes_2512[df_ocean_routes_2512["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2512[col] = pd.to_numeric(df_asian_2512[col])

df_asian_wash_2512 = df_asian_2512[df_asian_2512["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["port_full"].isin(coord.us_ports)]
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["ves_swt_mo"] > 0]
df_asian_wash_2512["distance"] = df_asian_wash_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2512["co2"] = df_asian_wash_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2512.to_csv("data/intermediate/PORTHS6MM_asian_wash_2512.csv")
df_asian_wash_2512.info()

<class 'pandas.DataFrame'>
Index: 73 entries, 824153 to 824435
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    73 non-null     object 
 1   cty_code     73 non-null     object 
 2   dist_unlade  73 non-null     object 
 3   port_unlade  73 non-null     object 
 4   year         73 non-null     object 
 5   month        73 non-null     object 
 6   gen_val_mo   73 non-null     int64  
 7   air_val_mo   73 non-null     int64  
 8   air_swt_mo   73 non-null     int64  
 9   ves_val_mo   73 non-null     int64  
 10  ves_swt_mo   73 non-null     int64  
 11  cnt_val_mo   73 non-null     int64  
 12  cnt_swt_mo   73 non-null     int64  
 13  gen_val_yr   73 non-null     object 
 14  air_val_yr   73 non-null     object 
 15  air_swt_yr   73 non-null     object 
 16  ves_val_yr   73 non-null     object 
 17  ves_swt_yr   73 non-null     object 
 18  cnt_val_yr   73 non-null     object 
 19  cnt_swt_yr   73 n

In [16]:
df_asian_wash_2512["port_full"].value_counts()

port_full
1003    6
1703    6
2704    6
1303    5
1801    5
2709    5
3002    5
1401    4
2811    4
1601    4
1803    4
5201    4
5301    4
3001    3
4909    3
1901    2
1001    1
2002    1
2904    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2024)

In [17]:
df_asian_tv_2412 = df_asian_2412[df_asian_2412["commodity"].str.startswith("8528")].copy()
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["port_full"].isin(coord.us_ports)]
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["ves_swt_mo"] > 0]
df_asian_tv_2412["distance"] = df_asian_tv_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2412.to_csv("data/intermediate/PORTHS6MM_asian_tv_2412.csv")
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

In [18]:
TvPortcodes2412 = df_asian_tv_2412["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2412)

port_full
0401     2
1001     2
1003    11
1101     1
1303     2
1401     6
1501     1
1601     6
1703     9
1801     2
1803     1
1816     2
1901     2
2704    15
2709    15
2809     2
2811     7
3001     5
3002     8
3604     2
4909     3
5201     6
5203     1
5301     4
5310     1
Name: count, dtype: int64


## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2025)

In [19]:
df_asian_tv_2512 = df_asian_2512[df_asian_2512["commodity"].str.startswith("8528")].copy()
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["port_full"].isin(coord.us_ports)]
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["ves_swt_mo"] > 0]
df_asian_tv_2512["distance"] = df_asian_tv_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2512["co2"] = df_asian_tv_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2512.to_csv("data/intermediate/PORTHS6MM_asian_tv_2512.csv")
df_asian_tv_2512.info()

<class 'pandas.DataFrame'>
Index: 111 entries, 978494 to 981139
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    111 non-null    object 
 1   cty_code     111 non-null    object 
 2   dist_unlade  111 non-null    object 
 3   port_unlade  111 non-null    object 
 4   year         111 non-null    object 
 5   month        111 non-null    object 
 6   gen_val_mo   111 non-null    int64  
 7   air_val_mo   111 non-null    int64  
 8   air_swt_mo   111 non-null    int64  
 9   ves_val_mo   111 non-null    int64  
 10  ves_swt_mo   111 non-null    int64  
 11  cnt_val_mo   111 non-null    int64  
 12  cnt_swt_mo   111 non-null    int64  
 13  gen_val_yr   111 non-null    object 
 14  air_val_yr   111 non-null    object 
 15  air_swt_yr   111 non-null    object 
 16  ves_val_yr   111 non-null    object 
 17  ves_swt_yr   111 non-null    object 
 18  cnt_val_yr   111 non-null    object 
 19  cnt_swt_yr   111

In [20]:
df_asian_tv_2512["port_full"].value_counts().sort_index()
TvPortcodes2512 = df_asian_tv_2512["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2512)

port_full
0401     2
1001     4
1003     9
1102     1
1303     2
1401     5
1601     5
1703     8
1801     2
1816     1
1901     3
2002     1
2704    16
2709    13
2809     2
2811     9
2904     1
3001     6
3002     4
3604     1
4909     2
5201     5
5203     2
5301     5
5310     2
Name: count, dtype: int64


So, upon filtering and cleaning data, we recognize that the vast majority of imports of Mexican washing machines into the USA is done through the land, rather than sea or plane. This confirmed my initial hypothesis on Mexico's transformation breakdown and now justifies the plan to calculate solely the inland leg's carbon footprint for imports from Mexico.

Now, we need to understand the distances from each state to state.

In [21]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Land-based Distance/Weight Data (Mexico, Pre-Tariff — Dec 2024)

In [22]:
df_land_imports_mex_2412 = df_land_imports_2412[df_land_imports_2412["cty_code"] == "2010"]
df_land_imports_mex_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
31,010121,2010,AZ,2024,12,0,0,0,0,0,...,0,0,44862,44862,0,0,0,0,0,0
32,010121,2010,CA,2024,12,0,0,0,0,0,...,0,0,33250,33250,0,0,0,0,0,0
33,010121,2010,FL,2024,12,0,0,0,0,0,...,0,0,29000,29000,29000,3500,0,0,0,0
34,010121,2010,NM,2024,12,0,0,0,0,0,...,0,0,3000,3000,0,0,0,0,0,0
35,010121,2010,TX,2024,12,0,0,0,0,0,...,0,0,232003,232003,0,0,0,0,0,0


In [23]:
df_land_imports_mex_v1_2412 = df_land_imports_mex_2412.T.drop_duplicates().T
df_land_imports_mex_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  object
 6   con_val_mo  39825 non-null  object
 7   air_val_mo  39825 non-null  object
 8   air_swt_mo  39825 non-null  object
 9   ves_val_mo  39825 non-null  object
 10  ves_swt_mo  39825 non-null  object
 11  cnt_val_mo  39825 non-null  object
 12  cnt_swt_mo  39825 non-null  object
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt_swt_yr  39825 n

In [24]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2412[col] = pd.to_numeric(df_land_imports_mex_v1_2412[col])
df_land_imports_mex_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  int64 
 6   con_val_mo  39825 non-null  int64 
 7   air_val_mo  39825 non-null  int64 
 8   air_swt_mo  39825 non-null  int64 
 9   ves_val_mo  39825 non-null  int64 
 10  ves_swt_mo  39825 non-null  int64 
 11  cnt_val_mo  39825 non-null  int64 
 12  cnt_swt_mo  39825 non-null  int64 
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt_swt_yr  39825 n

In [25]:
df_land_imports_mex_wash_2412 = df_land_imports_mex_v1_2412[df_land_imports_mex_v1_2412["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,0,0,28767168,28767168,0,0,40635,35100,40635,35100
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,0,0,5790782,5790782,0,0,0,0,0,0
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,328736,73407,19850931,19850931,0,0,3583492,831963,3583492,831963
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,0,0,16540032,16540032,0,0,0,0,0,0
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,0,0,13289441,13289441,0,0,0,0,0,0


In [26]:
df_land_imports_mex_wash_2412["state"].value_counts()

state
CA    2
CO    2
FL    2
GA    2
IN    2
KY    2
MD    2
OH    2
PR    2
TX    2
WA    2
NC    1
AZ    1
IL    1
MA    1
MI    1
MN    1
MO    1
NE    1
NJ    1
PA    1
Name: count, dtype: int64

In [27]:
df_land_imports_mex_wash_v1_2412 = df_land_imports_mex_wash_2412.copy()

df_land_imports_mex_wash_v1_2412["distance"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

Because the other dataset(ISTHS6MM) does not have land transportation weight and only land transportation value, I will calculate average value of washing machines and use it to find land transportation weight for Mexico. Because there are no rows with 0 for vessel-transported value or vessel-transported weight, I do not need to worry about rows with 0s affecting the mean.

In [28]:
washing_machine_price_coeff = df_land_imports_mex_wash_v1_2412["ves_val_mo"].mean()/df_land_imports_mex_wash_v1_2412["ves_swt_mo"].mean()

print(washing_machine_price_coeff, "$/kg")

4.694152410295217 $/kg


In [29]:
df_land_imports_mex_wash_v1_2412["gen_swt_mo"] = df_land_imports_mex_wash_v1_2412["gen_val_mo"] / washing_machine_price_coeff

In [30]:
df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-null     object 
 2

In [31]:
df_land_imports_mex_wash_v1_2412["co2"]

799500    1.361457e+08
799501    1.489580e+07
799502    5.556754e+07
799503    4.886573e+07
799504    5.251126e+07
799505    6.066518e+07
799506    2.078205e+08
799507    0.000000e+00
799508    0.000000e+00
799509    2.240968e+07
799510    6.027632e+07
799511    4.306294e+07
799700    4.739603e+05
799701    1.202557e+07
799702    2.384457e+06
799703    7.153023e+06
799704    1.743115e+06
799705    2.721901e+06
799706    4.576572e+05
799707    2.814080e+06
799708    0.000000e+00
799709    8.492860e+06
799710    0.000000e+00
799711    0.000000e+00
799712    0.000000e+00
799713    0.000000e+00
799714    2.385166e+06
799715    0.000000e+00
799716    3.582998e+07
799717    0.000000e+00
799718    2.467781e+08
799719    9.498459e+06
Name: co2, dtype: float64

In [32]:
df_land_imports_mex_wash_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_2412.csv")

## Land-based Distance/Weight Data (Mexico, Post-Tariff — Dec 2025)

In [33]:
df_land_imports_mex_2512 = df_land_imports_2512[df_land_imports_2512["cty_code"] == "2010"]
df_land_imports_mex_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
26,010121,2010,FL,2025,12,24000,24000,24000,2000,0,...,0,0,57000,57000,57000,5500,0,0,0,0
160,010129,2010,AZ,2025,12,0,0,0,0,0,...,0,0,275173,275173,0,0,0,0,0,0
161,010129,2010,CA,2025,12,0,0,0,0,0,...,0,0,504000,504000,504000,4500,0,0,0,0
162,010129,2010,FL,2025,12,51700,51700,51700,4000,0,...,0,0,143200,143200,143200,12957,0,0,0,0
163,010129,2010,NM,2025,12,0,0,0,0,0,...,0,0,195600,195600,0,0,0,0,0,0


In [34]:
df_land_imports_mex_v1_2512 = df_land_imports_mex_2512.T.drop_duplicates().T
df_land_imports_mex_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  object
 6   con_val_mo  39926 non-null  object
 7   air_val_mo  39926 non-null  object
 8   air_swt_mo  39926 non-null  object
 9   ves_val_mo  39926 non-null  object
 10  ves_swt_mo  39926 non-null  object
 11  cnt_val_mo  39926 non-null  object
 12  cnt_swt_mo  39926 non-null  object
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt_swt_yr  39926 n

In [35]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2512[col] = pd.to_numeric(df_land_imports_mex_v1_2512[col])
df_land_imports_mex_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  int64 
 6   con_val_mo  39926 non-null  int64 
 7   air_val_mo  39926 non-null  int64 
 8   air_swt_mo  39926 non-null  int64 
 9   ves_val_mo  39926 non-null  int64 
 10  ves_swt_mo  39926 non-null  int64 
 11  cnt_val_mo  39926 non-null  int64 
 12  cnt_swt_mo  39926 non-null  int64 
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt_swt_yr  39926 n

In [36]:
df_land_imports_mex_wash_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
849178,845011,2010,CA,2025,12,1807730,1807730,0,0,0,...,0,0,27563248,27563248,0,0,0,0,0,0
849179,845011,2010,CO,2025,12,398827,398827,0,0,0,...,0,0,4922868,4922868,0,0,0,0,0,0
849180,845011,2010,FL,2025,12,1378826,1378826,0,0,0,...,0,0,19121576,19121576,0,0,254100,58590,254100,58590
849181,845011,2010,GA,2025,12,764299,764299,0,0,0,...,0,0,13078900,13078900,0,0,0,0,0,0
849182,845011,2010,IL,2025,12,716040,716040,0,0,0,...,0,0,8086170,8086170,0,0,0,0,0,0


In [37]:
df_land_imports_mex_wash_v1_2512 = df_land_imports_mex_wash_2512.copy()

df_land_imports_mex_wash_v1_2512["distance"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_wash_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_wash_v1_2512["gen_val_mo"] / washing_machine_price_coeff)

df_land_imports_mex_wash_v1_2512["co2"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_wash_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_2512.csv")
df_land_imports_mex_wash_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 29 entries, 849178 to 849386
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   29 non-null     object 
 1   cty_code    29 non-null     object 
 2   state       29 non-null     object 
 3   year        29 non-null     object 
 4   month       29 non-null     object 
 5   gen_val_mo  29 non-null     int64  
 6   con_val_mo  29 non-null     int64  
 7   air_val_mo  29 non-null     int64  
 8   air_swt_mo  29 non-null     int64  
 9   ves_val_mo  29 non-null     int64  
 10  ves_swt_mo  29 non-null     int64  
 11  cnt_val_mo  29 non-null     int64  
 12  cnt_swt_mo  29 non-null     int64  
 13  gen_val_yr  29 non-null     object 
 14  con_val_yr  29 non-null     object 
 15  air_val_yr  29 non-null     object 
 16  air_swt_yr  29 non-null     object 
 17  ves_val_yr  29 non-null     object 
 18  ves_swt_yr  29 non-null     object 
 19  cnt_val_yr  29 non-null     object 
 2

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2024)

In [38]:
df_land_imports_mex_tv_2412 = df_land_imports_mex_v1_2412[
    df_land_imports_mex_v1_2412["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
942710,852842,2010,PR,2024,12,0,0,0,0,0,...,0,0,136900,136900,136900,1350,0,0,0,0
942737,852849,2010,PR,2024,12,0,0,0,0,0,...,0,0,162288,162288,162288,1972,0,0,0,0
942847,852852,2010,AL,2024,12,0,0,0,0,0,...,0,0,2290,2290,0,0,0,0,0,0
942848,852852,2010,AZ,2024,12,0,0,0,0,0,...,0,0,16250,16250,14000,297,0,0,0,0
942849,852852,2010,CA,2024,12,3059946,3059946,0,0,0,...,0,0,60472486,60472486,551557,1770,0,0,0,0


In [39]:
df_land_imports_mex_tv_2412[df_land_imports_mex_tv_2412["state"]=="VI"]

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
945083,852872,2010,VI,2024,12,0,0,0,0,0,...,0,0,8442,8442,0,0,8442,29,0,0


In [40]:
if not df_asian_tv_2412.empty and df_asian_tv_2412["ves_swt_mo"].sum() > 0:
    tv_price_coeff = df_asian_tv_2412["ves_val_mo"].mean() / df_asian_tv_2412["ves_swt_mo"].mean()
else:
    tv_price_coeff = 15.0  # fallback: ~$15/kg for TVs
print(tv_price_coeff, "$/kg")

17.303979051815883 $/kg


In [41]:
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_2412.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_v1_2412.drop(df_land_imports_mex_tv_v1_2412[df_land_imports_mex_tv_v1_2412["state"]=="VI"].index, inplace=False)

df_land_imports_mex_tv_v1_2412["distance"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2412["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2412["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2412["co2"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_tv_2412.csv")
df_land_imports_mex_tv_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 112 entries, 942710 to 945085
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   112 non-null    object 
 1   cty_code    112 non-null    object 
 2   state       112 non-null    object 
 3   year        112 non-null    object 
 4   month       112 non-null    object 
 5   gen_val_mo  112 non-null    int64  
 6   con_val_mo  112 non-null    int64  
 7   air_val_mo  112 non-null    int64  
 8   air_swt_mo  112 non-null    int64  
 9   ves_val_mo  112 non-null    int64  
 10  ves_swt_mo  112 non-null    int64  
 11  cnt_val_mo  112 non-null    int64  
 12  cnt_swt_mo  112 non-null    int64  
 13  gen_val_yr  112 non-null    object 
 14  con_val_yr  112 non-null    object 
 15  air_val_yr  112 non-null    object 
 16  air_swt_yr  112 non-null    object 
 17  ves_val_yr  112 non-null    object 
 18  ves_swt_yr  112 non-null    object 
 19  cnt_val_yr  112 non-null    object 
 

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2025)

In [42]:
df_land_imports_mex_tv_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
992995,852849,2010,CA,2025,12,0,0,0,0,0,...,0,0,28133,28133,0,0,0,0,0,0
993095,852852,2010,AL,2025,12,0,0,0,0,0,...,0,0,20107,20107,0,0,0,0,0,0
993096,852852,2010,AR,2025,12,74797,74797,0,0,0,...,0,0,141945,141945,0,0,0,0,0,0
993097,852852,2010,AZ,2025,12,0,0,0,0,0,...,0,0,38688,38688,0,0,0,0,0,0
993098,852852,2010,CA,2025,12,25343683,25343683,0,0,0,...,0,0,193420909,193420909,119413,279,0,0,0,0


In [43]:
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_2512.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_v1_2512.drop(df_land_imports_mex_tv_v1_2512[df_land_imports_mex_tv_v1_2512["state"]=="VI"].index, inplace=False)


df_land_imports_mex_tv_v1_2512["distance"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2512["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2512["co2"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_tv_2512.csv")
df_land_imports_mex_tv_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 124 entries, 992995 to 995325
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   124 non-null    object 
 1   cty_code    124 non-null    object 
 2   state       124 non-null    object 
 3   year        124 non-null    object 
 4   month       124 non-null    object 
 5   gen_val_mo  124 non-null    int64  
 6   con_val_mo  124 non-null    int64  
 7   air_val_mo  124 non-null    int64  
 8   air_swt_mo  124 non-null    int64  
 9   ves_val_mo  124 non-null    int64  
 10  ves_swt_mo  124 non-null    int64  
 11  cnt_val_mo  124 non-null    int64  
 12  cnt_swt_mo  124 non-null    int64  
 13  gen_val_yr  124 non-null    object 
 14  con_val_yr  124 non-null    object 
 15  air_val_yr  124 non-null    object 
 16  air_swt_yr  124 non-null    object 
 17  ves_val_yr  124 non-null    object 
 18  ves_swt_yr  124 non-null    object 
 19  cnt_val_yr  124 non-null    object 
 

In [ ]:
# Mexico washing machines — Dec 2024 (PORTHS6MM)
df_mex_wash_2412 = df_ocean_routes_2412[
    (df_ocean_routes_2412["cty_code"] == "2010") &
    (df_ocean_routes_2412["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2412["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2412[col] = pd.to_numeric(df_mex_wash_2412[col], errors="coerce").fillna(0)

df_mex_wash_2412 = df_mex_wash_2412[df_mex_wash_2412["gen_val_mo"] > 0]

df_mex_wash_2412["gen_swt_mo"] = df_mex_wash_2412["ves_swt_mo"].where(
    df_mex_wash_2412["ves_swt_mo"] > 0,
    df_mex_wash_2412["gen_val_mo"] / washing_machine_price_coeff
)
df_mex_wash_2412["distance"] = df_mex_wash_2412.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2412["co2"] = df_mex_wash_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_wash_2412.info()

In [ ]:
# Mexico washing machines — Dec 2025 (PORTHS6MM)
df_mex_wash_2512 = df_ocean_routes_2512[
    (df_ocean_routes_2512["cty_code"] == "2010") &
    (df_ocean_routes_2512["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2512["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2512[col] = pd.to_numeric(df_mex_wash_2512[col], errors="coerce").fillna(0)

df_mex_wash_2512 = df_mex_wash_2512[df_mex_wash_2512["gen_val_mo"] > 0]

df_mex_wash_2512["gen_swt_mo"] = df_mex_wash_2512["ves_swt_mo"].where(
    df_mex_wash_2512["ves_swt_mo"] > 0,
    df_mex_wash_2512["gen_val_mo"] / washing_machine_price_coeff
)
df_mex_wash_2512["distance"] = df_mex_wash_2512.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2512["co2"] = df_mex_wash_2512.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_wash_2512.info()

In [ ]:
# Mexico TVs — Dec 2024 (PORTHS6MM)
df_mex_tv_2412 = df_ocean_routes_2412[
    (df_ocean_routes_2412["cty_code"] == "2010") &
    (df_ocean_routes_2412["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2412["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2412[col] = pd.to_numeric(df_mex_tv_2412[col], errors="coerce").fillna(0)

df_mex_tv_2412 = df_mex_tv_2412[df_mex_tv_2412["gen_val_mo"] > 0]

df_mex_tv_2412["gen_swt_mo"] = df_mex_tv_2412["ves_swt_mo"].where(
    df_mex_tv_2412["ves_swt_mo"] > 0,
    df_mex_tv_2412["gen_val_mo"] / tv_price_coeff
)
df_mex_tv_2412["distance"] = df_mex_tv_2412.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2412["co2"] = df_mex_tv_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_tv_2412.info()

In [ ]:
# Mexico TVs — Dec 2025 (PORTHS6MM)
df_mex_tv_2512 = df_ocean_routes_2512[
    (df_ocean_routes_2512["cty_code"] == "2010") &
    (df_ocean_routes_2512["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2512["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2512[col] = pd.to_numeric(df_mex_tv_2512[col], errors="coerce").fillna(0)

df_mex_tv_2512 = df_mex_tv_2512[df_mex_tv_2512["gen_val_mo"] > 0]

df_mex_tv_2512["gen_swt_mo"] = df_mex_tv_2512["ves_swt_mo"].where(
    df_mex_tv_2512["ves_swt_mo"] > 0,
    df_mex_tv_2512["gen_val_mo"] / tv_price_coeff
)
df_mex_tv_2512["distance"] = df_mex_tv_2512.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2512["co2"] = df_mex_tv_2512.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_tv_2512.info()

## Weight Data(USA Trade Online) - irrelevant?

Already well-organized, the weight data does not need much cleaning.

However, the previous datasets(US Census) contains everything necessary. Not just weight of imports, but also origin and destination info. Use this as a sanity check for weight of ocean-side imports.

In [44]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [45]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [46]:
washing_machine_df_v1[washing_machine_df_v1["Country"]=="Mexico"].head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
32,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,January 2025,0.0,55246.0
33,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,February 2025,0.0,74445.0
34,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,March 2025,0.0,239852.0
35,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,April 2025,0.0,175512.0
36,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,May 2025,0.0,199598.0


In [47]:
washing_machine_df_v1["Time"].value_counts()

Time
April 2025            9
June 2025             9
July 2025             9
October 2025          9
December 2025         9
2026 through March    9
January 2026          9
January 2025          8
February 2025         8
March 2025            8
May 2025              8
August 2025           8
September 2025        8
November 2025         8
February 2026         8
March 2026            8
Name: count, dtype: int64

In [48]:
washing_machine_df_v1["Air SWT (kg)"].mean()

np.float64(171.28148148148148)

# CO2 Calculations(transportation)

Now that we have the three necessary components - ton-km CO2 factor, weight of imports and distance of imports, then we can calculate the carbon footprint of transportation. This is only one step, as we also need to calculate CO2 of manufacturing(grid intensity x energy spent on manufacturing) in order to understand the full extent of the carbon footprint.

CO2, in this case, is measured in grams. For now, we do this only for 5 countries(China, Vietnam, South Korea, India, Mexico) and only for the month of January 2025, using the datasets taken from census.gov website.

In [49]:
# def compute_co2(row, transportation_type):
#     weight = row[transportation_type]
#     dist = row["distance"]
#     co2_coeff = ton_km[row["cty_code"]]
#     return weight * dist * co2_coeff

df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args = ["gen_swt_mo"], axis=1)

In [50]:
df_land_imports_mex_wash_v1_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,distance,gen_swt_mo,co2
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,28767168,0,0,40635,35100,40635,35100,2847.115733,597735.172349,1.361457e+08
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,5790782,0,0,0,0,0,0,2028.894380,91772.904317,1.489580e+07
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,19850931,0,0,3583492,831963,3583492,831963,2442.374793,284392.981590,5.556754e+07
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,16540032,0,0,0,0,0,0,2389.453884,255632.304858,4.886573e+07
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,13289441,0,0,0,0,0,0,2660.910286,246679.037830,5.251126e+07


In [51]:
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-null     object 
 2

In [52]:
# df_asian_routes_wash["co2"] = df_asian_routes_wash["vessel_swt_mo"]*df_asian_routes_wash["distance"]*ton_km[df_asian_routes_wash["cty_code"]]
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args = ["ves_swt_mo"], axis=1)


In [53]:
df_asian_tv_2412.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full,distance,co2
953212,852849,5700,27,09,2024,12,136068,0,0,136068,...,174835,0,0,174835,27408,174835,27408,2709,10674.961095,8.998138e+05
953794,852852,5520,17,03,2024,12,626316,0,0,626316,...,2608368,0,0,2608368,79266,2608368,79266,1703,20949.030244,3.620474e+06
953797,852852,5520,27,04,2024,12,35278490,0,0,35278490,...,425218271,0,0,425218271,23321912,422194514,23231547,2704,13525.167789,2.115113e+08
953798,852852,5520,27,09,2024,12,1250476,0,0,1250476,...,52803271,0,0,52803271,2479649,52803271,2479649,2709,13525.167789,3.925464e+06
953803,852852,5520,30,02,2024,12,1117640,0,0,1117640,...,3321417,0,0,3321417,114877,3321417,114877,3002,12411.976275,3.666411e+06


In [54]:
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

# Statistics

Now we use the difference-in-difference method to evaluate the extent, to which the tariffs have changed the CO2 footprint. We compare the differences in the control group and the treatment group. 

What will be the control group in our case? 

The control group will be "commodity", more specifically TV sets. TVs have mostly the same demand as washing machines, since both are bought together when people move to a new home. However, unlike the washing machines, TV sets don't fall under the Liberation Day tariffs and steel tariffs of 2025, which is due to goods with semiconductors being exempt from tariffs. These factors make TVs a good control group, since they're similar to washing machines in most key regards, except for the thing that we try to measure.

However, TV sets are lighter than washing machines, so CO2 change scales differently. One way to fix this is to normalize CO2 data by calculating CO2/kg of commodity, rather than total CO2. 


This code sets up a DiD model and runs an OLS regression combining **both Mexican and Asian data**. Mexican observations are grouped by destination state; Asian observations are grouped by origin country and destination port.

Then it creates a long table, in which each row is a combination of country, date and product(20 rows in total). Each row has total weight of products imported, total CO2 released and weighted average distance. Each of the aforementioned

In [ ]:
def _tag(df, commodity, period, weight_col="ves_swt_mo"):
    out = df[["cty_code", weight_col, "co2", "distance", "port_full"]].copy()
    out = out.rename(columns={weight_col: "weight"})
    out["commodity"] = commodity
    out["period"] = period
    return out

df_sea_all = pd.concat([
    _tag(df_asian_wash_2412, "washing_machine", "2412"),
    _tag(df_asian_wash_2512, "washing_machine", "2512"),
    _tag(df_asian_tv_2412,   "tv",              "2412"),
    _tag(df_asian_tv_2512,   "tv",              "2512"),
    _tag(df_mex_wash_2412,  "washing_machine", "2412", weight_col="gen_swt_mo"),
    _tag(df_mex_wash_2512,  "washing_machine", "2512", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2412,    "tv",              "2412", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2512,    "tv",              "2512", weight_col="gen_swt_mo"),
], ignore_index=True)

df_sea_all["dist_x_weight"] = df_sea_all["distance"] * df_sea_all["weight"]

df_sea_agg = df_sea_all.groupby(["cty_code", "commodity", "period", "port_full"]).agg(
    total_weight=("weight", "sum"),
    total_co2=("co2", "sum"),
    _dist_x_wt=("dist_x_weight", "sum"),
).reset_index()

df_sea_agg["avg_distance"] = df_sea_agg["_dist_x_wt"] / df_sea_agg["total_weight"]
df_sea_agg = df_sea_agg.drop(columns="_dist_x_wt")
df_sea_agg

In [ ]:
import statsmodels.api as sm

df_did = df_sea_agg[df_sea_agg["total_weight"] > 0].copy()
df_did["co2_intensity"] = df_did["total_co2"] / df_did["total_weight"]
df_did["treated"] = (df_did["commodity"] == "washing_machine").astype(int)
df_did["post"]    = (df_did["period"]    == "2512").astype(int)
df_did["s"] = df_did["cty_code"].map(coord.tariff_rate) * df_did["treated"]
df_did["treated_x_post"] = df_did["s"] * df_did["post"]

df_did[["cty_code", "port_full", "co2_intensity", "treated", "post", "treated_x_post"]].sort_values("co2_intensity", ascending=False).head(10)

In [64]:
df_sea_all.head()

,cty_code,ves_swt_mo,co2,distance,port_unlade,commodity,period,dist_x_weight
0,5520,223317,3.130660e+07,20027.006554,03,washing_machine,2412,4.472371e+09
1,5520,2683,3.866830e+05,20589.054455,03,washing_machine,2412,5.524043e+07
2,5520,4488,6.581347e+05,20949.030244,03,washing_machine,2412,9.401925e+07
3,5520,13663,2.034897e+06,21276.406623,01,washing_machine,2412,2.906995e+08
4,5520,89496,8.473139e+06,13525.167789,04,washing_machine,2412,1.210448e+09


A sanity check. If std = 0, then tinkering with it is useless. For land, it's 0, for sea it's 9.4

In [ ]:
# CO2 intensity variation by origin country — a non-zero std means the DiD has signal to work with
df_did.groupby(["cty_code", "commodity"])["co2_intensity"].std().describe()

In [ ]:
sea_check = df_sea_agg.copy()
sea_check["co2_intensity"] = sea_check["total_co2"] / sea_check["total_weight"]

sea_check.groupby(["port_full", "commodity"])["co2_intensity"].std().describe()

It makes sense for carbon footprint of imports to California, Georgia and Texas to be high, since those states border Mexico. However, why do imports of TVs from Mexico to New Jersey have such high total value and CO2 footprint?

In [57]:
X = sm.add_constant(df_did[["treated", "post", "treated_x_post"]])
model = sm.OLS(df_did["co2_intensity"], X, missing="drop").fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          co2_intensity   R-squared:                       0.033
Model:                            OLS   Adj. R-squared:                 -0.289
Method:                 Least Squares   F-statistic:                   0.07589
Date:                Sun, 26 Jul 2026   Prob (F-statistic):              0.971
Time:                        16:08:18   Log-Likelihood:                -61.042
No. Observations:                  13   AIC:                             130.1
Df Residuals:                       9   BIC:                             132.3
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             65.7885     21.449      3.

Having run the OLS regression on the difference-in-differences model, we found that for all countries' imports, there is no difference in the CO2 before and after the tariffs. 

Why could that be? I know that for land transportation to the USA, the number of trade routes is more limited compared to ocean routes, so there is little that Mexico can do to change their trade routes in the aftermath of tariffs. But how much did the number of imported goods change?

Also, how do I interpret those other statistics? F-statistic, log-likelihood, AIC/BIC, Durbin-Watson, et cetera.